# Whisper testing

This notebook is used to compare Whisper efficiency and accuracy based on the model used (small, base or tiny) and the quantization applied (int8). During end-to-end testing of the application pipeline, latency is prohibitive, with Whisper being one of the components contributing the most to it with an average latency of 2.919 seconds per sentence passed. 

The dataset used to determine the bast Whisper variant is espnet / yodas-granary from Hugging Face (https://huggingface.co/datasets/espnet/yodas-granary). This is a modified version of the larger nvidia/Granary dataset, specifically designed for ASR across 23 languages. The transcriptions of the audio clips have been derived faster-whisper-large-v3 model. Given the size and complexity of this model, most transcriptions are expected to be correct. However, in the future the accuracy of the dataset should be examined. Given that for now the application only supports English, the dataset is filtered to extract the entries in English. If we decide to extend to other languages later, this dataset will be helpful in testing Whisper in other languages as well.

## Loading

In [ ]:
!pip install --upgrade pip
!pip install --upgrade transformers accelerate datasets[audio]
!pip install flash-attn --no-build-isolation
!pip install --upgrade optimum

In [ ]:
from datasets import 
import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline

In [ ]:
ds = load_dataset("espnet/yodas-granary", "English")            # Load only english

## Filtering

In [ ]:
filtered_ds = ds[ds['duration'] < 30]

## Testing


To test the affect of changing the Whisper model variant in accuracy and performance the following models are used:
1. faster-whisper small
2. faster-whisper base
3. faster-whisper tiny
4. distil-small
5. distil-large
5. turbo whisper

The last 3 models are distilled versions of Whisper. On the one hand, distil-small and distil-large were trained by freezing the encoder layers and using only the first and the last decoder layers from the original whisper model, discarding the rest. According to the creators, distiled Whisper is 6 times faster, 49% smaller and performs within 1% WER of the original model. Moreover, it is more robust to noise and hallucinations. On the other hand, turbo whisper is inspired from distil whisper and it is an optimization of whipser-large-v3. It hs only 4 decoder layers instead of 32, but instead of using distillation, the model is trained for two more epochs over the same amount of data as large-v3. To evaluate the models above, 2 metrics are used:

* WER : calculates substitutions, deletions and insertions on word level measuring the accuracy of the model
* RTFx (Inverse real time factor) : ratio of duration divided by processing time measuring the latency of the model

### Assumption

The assumption used during testing is that the audio clips are shorter than 30 seconds, which is the maximum receptive field of the Whisper models. It is crucial that the possibility of larger audio clips provided by the user is considered. In this case, chunking is needed to process the audio clips. In the first round of testing with no quantization, the precision used is float16.

### Optimizations

To make sure that the capabilities of the T4 GPUs provided by Google Collab and of Whisper are fully exploited, the following optimizations are performed:

1) Optimum : Hugging Face library supporting the ONNX Runtime model accelerator. This uses optimization techniques that fuse common operations inso a single node and constant folding to reduce the number of computations needed, placing the most computationally expensive operations on the GPU.

2) Flash attention : this is not supported on the Google Collab T4s, but will be used in the EC2 instance. It can significantly speed up inference by parallelizing the attention computation over sequence length and partitioning the work between GPU threads to reduce communication. If flash attention is not supported by the GPU (ie in Google Collab) torch scale-product-attention (SDPA) is used to convert the whisper models to "better transformers".

In [ ]:
device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32
print(f"Device : {device} \n")

In [ ]:
distilLarge = AutoModelForSpeechSeq2Seq.from_pretrained(
    "distil-whisper/distil-large-v3", 
    torch_dtype=torch_dtype, 
    low_cpu_mem_usage=True, 
    use_safetensors=True,
    use_flash_attention_2=True,     # Flash attention
)
distilLarge = distilLarge.to_bettertransformer()            # SPDA
distilLarge.to(device)


distilSmall = AutoModelForSpeechSeq2Seq.from_pretrained(
    "distil-whisper/distil-small.en", 
    torch_dtype=torch_dtype, 
    low_cpu_mem_usage=True, 
    use_safetensors=True
)
distilSmall = distilSmall.to_bettertransformer()            # SPDA
distilSmall.to(device)

processorLarge = AutoProcessor.from_pretrained("distil-whisper/distil-large-v3")
processorSmall = AutoProcessor.from_pretrained("distil-whisper/distil-small.en")

pipeLarge = pipeline(
    "automatic-speech-recognition",
    model=distilLarge,
    tokenizer=processorLarge.tokenizer,
    feature_extractor=processorLarge.feature_extractor,
    max_new_tokens=128,
    torch_dtype=torch_dtype,
    device=device,
)

pipeSmall = pipeline(
    "automatic-speech-recognition",
    model=distilSmall,
    tokenizer=processorSmall.tokenizer,
    feature_extractor=processorSmall.feature_extractor,
    max_new_tokens=128,
    torch_dtype=torch_dtype,
    device=device,
)


# Quantization

After the optimal model has been found from the above with the optimization techniques outlined, quantization is performed to deduce whether the loss in accuracy is small enough when compared to the latency reduction. More specifically, 8-bit quantization is used with the python libraries bitsandytes and accelerate